# Grad-CAM 

Este notebook:
- Carga un **modelo preentrenado de torchvision** (ResNet50).
- Aplica **Grad-CAM** a **todas las imágenes** en una carpeta.
- Guarda resultados en `output_gradcam/` como PNG (original, heatmap y overlay).

In [3]:
# (Opcional) Si te falta algo en tu entorno, descomenta e instala:
# !pip install -U torch torchvision pillow matplotlib numpy

import os
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Config de matplotlib
plt.rcParams["figure.dpi"] = 120

if torch.cuda.is_available():
    device = torch.device("cuda")      # o "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")       # Apple Silicon
else:
    device = torch.device("cpu")


print("Device:", device)


Device: mps


In [4]:
# Utilidades: cargar imagen + overlay con heatmap

def load_image_pil(path: str, size=224):
    """Carga una imagen y devuelve (pil_rgb, tensor_1x3xHxW) normalizado tipo ImageNet."""
    pil = Image.open(path).convert("RGB")
    pil_resized = pil.resize((size, size))

    # Normalización ImageNet (para modelos torchvision preentrenados)
    img = np.array(pil_resized).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_norm = (img - mean) / std

    # HWC -> CHW
    tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0)
    return pil_resized, tensor.to(device)

def normalize_0_1(x: np.ndarray, eps=1e-8):
    x = x - x.min()
    x = x / (x.max() + eps)
    return x

def overlay_heatmap_on_image(pil_img: Image.Image, cam_0_1: np.ndarray, alpha=0.45):
    """Devuelve (heatmap_rgb_uint8, overlay_rgb_uint8)."""
    # cam_0_1: HxW en [0,1]
    cam_0_1 = np.clip(cam_0_1, 0, 1)

    # Colormap
    cmap = plt.get_cmap("jet")
    heat = cmap(cam_0_1)[:, :, :3]  # HxWx3, float [0,1]

    img = np.array(pil_img).astype(np.float32) / 255.0

    overlay = (1 - alpha) * img + alpha * heat
    overlay = np.clip(overlay, 0, 1)

    heat_u8 = (heat * 255).astype(np.uint8)
    overlay_u8 = (overlay * 255).astype(np.uint8)
    return heat_u8, overlay_u8


In [5]:
# Cargar modelo torchvision (ResNet50 preentrenado)

try:
    from torchvision.models import resnet50, ResNet50_Weights
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    # Labels de ImageNet (para imprimir top-1). Si no existen, no pasa nada.
    imagenet_labels = weights.meta.get("categories", None)
except Exception:
    # Fallback para versiones antiguas
    from torchvision.models import resnet50
    model = resnet50(pretrained=True)
    imagenet_labels = None

model = model.to(device).eval()

# Capa objetivo típica para ResNet: último bloque conv
target_layer = model.layer4[-1]

print("Modelo listo:", model.__class__.__name__)
print("Target layer:", target_layer.__class__.__name__)


Modelo listo: ResNet
Target layer: Bottleneck


In [6]:
# Implementación Grad-CAM (clasificación)

class GradCAM:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            # grad_out es tupla; grad_out[0] es gradiente w.r.t. out
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    @torch.no_grad()
    def predict(self, x):
        logits = self.model(x)
        probs = F.softmax(logits, dim=1)
        return logits, probs

    def cam(self, x, class_idx=None):
        """Devuelve (cam_0_1, class_idx, score)."""
        self.model.zero_grad(set_to_none=True)

        logits = self.model(x)
        if class_idx is None:
            class_idx = int(torch.argmax(logits, dim=1).item())

        score = logits[0, class_idx]
        score.backward(retain_graph=True)

        # activations: [1, C, H, W], gradients: [1, C, H, W]
        A = self.activations
        dA = self.gradients

        # Pesos por canal: promedio espacial del gradiente
        weights = dA.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]
        cam = (weights * A).sum(dim=1, keepdim=False)  # [1, H, W]
        cam = F.relu(cam)

        cam_np = cam[0].detach().cpu().numpy()

        # Reescalar a tamaño input (224x224) si hace falta
        cam_np = np.array(Image.fromarray(cam_np).resize((x.shape[-1], x.shape[-2])))
        cam_np = normalize_0_1(cam_np)

        return cam_np, class_idx, float(score.detach().cpu().item())

gc = GradCAM(model, target_layer)
print("GradCAM inicializado.")


GradCAM inicializado.


In [ ]:
# CONFIG: carpeta de entrada y salida

# Cambia esto a tu carpeta con imágenes (jpg/png/jpeg)
   
INPUT_DIR = Path("../imagenes/original")  
OUTPUT_DIR = Path("../imagenes/output_gradcam")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Opcional: limitar cuántas procesa (None = todas)
MAX_IMAGES = None

print("INPUT_DIR:", INPUT_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


INPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/original
OUTPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/output_gradcam


In [8]:
# Ejecutar Grad-CAM sobre todas las imágenes de la carpeta

paths = [p for p in sorted(INPUT_DIR.rglob("*")) if p.suffix.lower() in EXTS]
if MAX_IMAGES is not None:
    paths = paths[:MAX_IMAGES]

print("Imágenes encontradas:", len(paths))
if len(paths) == 0:
    raise FileNotFoundError(f"No encontré imágenes en {INPUT_DIR}. Revisa INPUT_DIR.")

for i, p in enumerate(paths, 1):
    pil_img, x = load_image_pil(str(p), size=224)

    # Predicción + CAM sobre clase predicha
    cam_0_1, cls_idx, score = gc.cam(x, class_idx=None)

    # Etiqueta top-1 si está disponible
    if imagenet_labels is not None and 0 <= cls_idx < len(imagenet_labels):
        cls_name = imagenet_labels[cls_idx]
    else:
        cls_name = str(cls_idx)

    heat_u8, overlay_u8 = overlay_heatmap_on_image(pil_img, cam_0_1, alpha=0.45)

    # Guardar figura comparativa
    out_name = OUTPUT_DIR / f"{p.stem}_gradcam.png"
    fig = plt.figure(figsize=(9, 3))

    ax1 = plt.subplot(1, 3, 1)
    ax1.imshow(pil_img)
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = plt.subplot(1, 3, 2)
    ax2.imshow(heat_u8)
    ax2.set_title("Heatmap")
    ax2.axis("off")

    ax3 = plt.subplot(1, 3, 3)
    ax3.imshow(overlay_u8)
    ax3.set_title(f"Overlay\n{cls_name} (logit={score:.2f})")
    ax3.axis("off")

    plt.tight_layout()
    fig.savefig(out_name, bbox_inches="tight")
    plt.close(fig)

    if i % 10 == 0 or i == len(paths):
        print(f"[{i}/{len(paths)}] Guardado: {out_name}")

print("Revisa la carpeta output_gradcam/")


Imágenes encontradas: 11
[10/11] Guardado: ../imagenes/output_gradcam/image10_gradcam.png
[11/11] Guardado: ../imagenes/output_gradcam/image11_gradcam.png
Revisa la carpeta output_gradcam/
